In [0]:
from pyspark.sql import functions as F

# 1. Read Bronze table
bronze_df = spark.table(
    "workspace.retail_warehouse.bronze_retail_transactions"
)

# 2. Standardize data types
silver_df = (
    bronze_df
    .withColumn("invoice", F.trim(F.col("invoice").cast("string")))
    .withColumn("stockcode", F.trim(F.col("stockcode").cast("string")))
    .withColumn("description", F.trim(F.col("description")))
    .withColumn("quantity", F.col("quantity").cast("integer"))
    .withColumn(
        "invoicedate",
        F.coalesce(
            F.to_timestamp("invoicedate", "M/d/yyyy H:mm"),
            F.to_timestamp("invoicedate", "M/d/yyyy H:mm:ss"),
            F.to_timestamp("invoicedate", "yyyy-MM-dd HH:mm:ss"),
            F.col("invoicedate").cast("timestamp")
        )
    )
    .withColumn("price", F.col("price").cast("decimal(18,2)"))
    .withColumn("customer_id", F.col("customer_id").cast("long"))
    .withColumn("country", F.trim(F.col("country")))
)

# 3. Remove exact duplicate business records
business_columns = [
    "invoice", "stockcode", "description", "quantity",
    "invoicedate", "price", "customer_id", "country"
]

before_dedup = silver_df.count()

silver_df = silver_df.dropDuplicates(business_columns)

after_dedup = silver_df.count()

# 4. Identify invalid records
invalid_condition = (
    F.col("invoice").isNull() |
    (F.col("invoice") == "") |
    F.col("stockcode").isNull() |
    (F.col("stockcode") == "") |
    F.col("quantity").isNull() |
    (F.col("quantity") == 0) |
    F.col("invoicedate").isNull() |
    F.col("price").isNull() |
    (F.col("price") < 0)
)

invalid_count = silver_df.filter(invalid_condition).count()

# 5. Keep valid transactions
silver_df = silver_df.filter(~invalid_condition)

# 6. Add business transformation columns
silver_df = (
    silver_df
    .withColumn(
        "transaction_type",
        F.when(
            (F.col("invoice").startswith("C")) |
            (F.col("quantity") < 0),
            "RETURN_OR_CANCELLATION"
        ).otherwise("SALE")
    )
    .withColumn(
        "total_amount",
        F.round(F.col("quantity") * F.col("price"), 2)
    )
    .withColumn("invoice_year", F.year("invoicedate"))
    .withColumn("invoice_month", F.month("invoicedate"))
    .withColumn("silver_processed_timestamp", F.current_timestamp())
)

# 7. Save Silver Delta table
silver_table = (
    "workspace.retail_warehouse.silver_retail_transactions"
)

(
    silver_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(silver_table)
)

# 8. Print data quality report
print("=" * 55)
print("SILVER TRANSFORMATION COMPLETED")
print("=" * 55)

print("Bronze records:", before_dedup)
print("Duplicates removed:", before_dedup - after_dedup)
print("Invalid records:", invalid_count)
print("Silver records:", silver_df.count())

display(silver_df.limit(10))

SILVER TRANSFORMATION COMPLETED
Bronze records: 1067371
Duplicates removed: 34335
Invalid records: 5
Silver records: 1033031


invoice,stockcode,description,quantity,invoicedate,price,customer_id,country,ingestion_timestamp,source_file,transaction_type,total_amount,invoice_year,invoice_month,silver_processed_timestamp
489436,21755,LOVE BUILDING BLOCK WORD,18,2009-12-01T09:06:00.000Z,5.45,13078,United Kingdom,2026-11-08T10:56:02.740Z,dbfs:/Volumes/workspace/retail_warehouse/raw_data/year_2009-2010.csv,SALE,98.10,2009,12,2026-11-08T10:58:28.611Z
489436,22142,CHRISTMAS CRAFT WHITE FAIRY,12,2009-12-01T09:06:00.000Z,1.45,13078,United Kingdom,2026-11-08T10:56:02.740Z,dbfs:/Volumes/workspace/retail_warehouse/raw_data/year_2009-2010.csv,SALE,17.40,2009,12,2026-11-08T10:58:28.611Z
489437,20703,BLUE PADDED SOFT MOBILE,3,2009-12-01T09:08:00.000Z,4.25,15362,United Kingdom,2026-11-08T10:56:02.740Z,dbfs:/Volumes/workspace/retail_warehouse/raw_data/year_2009-2010.csv,SALE,12.75,2009,12,2026-11-08T10:58:28.611Z
489438,84031A,CHARLIE+LOLA RED HOT WATER BOTTLE,56,2009-12-01T09:24:00.000Z,3.00,18102,United Kingdom,2026-11-08T10:56:02.740Z,dbfs:/Volumes/workspace/retail_warehouse/raw_data/year_2009-2010.csv,SALE,168.00,2009,12,2026-11-08T10:58:28.611Z
489438,84519B,CARROT CHARLIE+LOLA COASTER SET,60,2009-12-01T09:24:00.000Z,2.40,18102,United Kingdom,2026-11-08T10:56:02.740Z,dbfs:/Volumes/workspace/retail_warehouse/raw_data/year_2009-2010.csv,SALE,144.00,2009,12,2026-11-08T10:58:28.611Z
489438,85183A,CHARLIE & LOLA WASTEPAPER BIN BLUE,60,2009-12-01T09:24:00.000Z,2.40,18102,United Kingdom,2026-11-08T10:56:02.740Z,dbfs:/Volumes/workspace/retail_warehouse/raw_data/year_2009-2010.csv,SALE,144.00,2009,12,2026-11-08T10:58:28.611Z
489439,22064,PINK DOUGHNUT TRINKET POT,12,2009-12-01T09:28:00.000Z,1.65,12682,France,2026-11-08T10:56:02.740Z,dbfs:/Volumes/workspace/retail_warehouse/raw_data/year_2009-2010.csv,SALE,19.80,2009,12,2026-11-08T10:58:28.611Z
489439,20749,ASSORTED COLOUR MINI CASES,2,2009-12-01T09:28:00.000Z,7.95,12682,France,2026-11-08T10:56:02.740Z,dbfs:/Volumes/workspace/retail_warehouse/raw_data/year_2009-2010.csv,SALE,15.90,2009,12,2026-11-08T10:58:28.611Z
489441,22321,BIRD DECORATION RED SPOT,12,2009-12-01T09:44:00.000Z,0.72,18087,United Kingdom,2026-11-08T10:56:02.740Z,dbfs:/Volumes/workspace/retail_warehouse/raw_data/year_2009-2010.csv,SALE,8.64,2009,12,2026-11-08T10:58:28.611Z
489442,22091,EMPIRE TISSUE BOX,12,2009-12-01T09:46:00.000Z,1.25,13635,United Kingdom,2026-11-08T10:56:02.740Z,dbfs:/Volumes/workspace/retail_warehouse/raw_data/year_2009-2010.csv,SALE,15.00,2009,12,2026-11-08T10:58:28.611Z


In [0]:
silver = spark.table(
    "workspace.retail_warehouse.silver_retail_transactions"
)

print("SILVER TABLE VALIDATION")

silver.printSchema()

display(
    silver.groupBy("transaction_type")
    .count()
)

display(
    silver.select(
        F.count("*").alias("total_records"),
        F.countDistinct("invoice").alias("unique_invoices"),
        F.countDistinct("customer_id").alias("unique_customers"),
        F.sum("total_amount").alias("total_sales_amount"),
        F.sum(F.col("customer_id").isNull().cast("int"))
         .alias("missing_customer_ids")
    )
)

SILVER TABLE VALIDATION
root
 |-- invoice: string (nullable = true)
 |-- stockcode: string (nullable = true)
 |-- description: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- invoicedate: timestamp (nullable = true)
 |-- price: decimal(18,2) (nullable = true)
 |-- customer_id: long (nullable = true)
 |-- country: string (nullable = true)
 |-- ingestion_timestamp: timestamp (nullable = true)
 |-- source_file: string (nullable = true)
 |-- transaction_type: string (nullable = true)
 |-- total_amount: decimal(30,2) (nullable = true)
 |-- invoice_year: integer (nullable = true)
 |-- invoice_month: integer (nullable = true)
 |-- silver_processed_timestamp: timestamp (nullable = true)



transaction_type,count
SALE,1010534
RETURN_OR_CANCELLATION,22497


total_records,unique_invoices,unique_customers,total_sales_amount,missing_customer_ids
1033031,53623,5942,19014209.82,235146
